# Relatório Semanal de Investimentos — Estratégia de Dividendos

**Como usar:**
1. Vá em `Editar → Configurações do notebook` e autorize o Google Drive quando solicitado
2. Adicione sua chave Gemini nos Secrets (ícone de cadeado 🔑 na barra lateral): nome `GEMINI_API_KEY`
3. Execute todas as células: `Ctrl+F9` ou `Runtime → Run all`
4. O arquivo `.docx` será salvo em `Meu Drive/relatorios_besst/` e iniciará o download automático

> Este notebook gera um relatório fundamentalista das melhores ações dos setores de renda passiva listadas no Ibovespa.

In [ ]:
# CÉLULA 1 — Instalação das bibliotecas
# Executar apenas uma vez por sessão do Colab.
print('Instalando bibliotecas...')
import subprocess
subprocess.run(['pip', 'install', '-q', 'python-docx', 'google-genai'], check=True)
print('Bibliotecas instaladas!')

In [ ]:
# CÉLULA 2 — Conexão com o Google Drive
# Ao executar, o Colab pedirá permissão de acesso — autorize.
from google.colab import drive
drive.mount('/content/drive')
import os
PASTA_DRIVE = '/content/drive/MyDrive/relatorios_besst'
os.makedirs(PASTA_DRIVE, exist_ok=True)
os.makedirs('/content/dados', exist_ok=True)
print(f'Drive conectado! Relatórios serão salvos em: {PASTA_DRIVE}')

In [ ]:
# CÉLULA 3 — Leitura da chave Gemini dos Secrets
# Como adicionar:
#   1. Clique no ícone de cadeado na barra lateral esquerda
#   2. Clique em '+ Adicionar novo secret'
#   3. Nome: GEMINI_API_KEY | Valor: sua_chave_aqui
#   4. Ative o acesso ao notebook
from google.colab import userdata
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print(f'Chave Gemini carregada: {GEMINI_API_KEY[:6]}...{GEMINI_API_KEY[-4:]}')
except Exception:
    print('GEMINI_API_KEY nao encontrada — relatório sem análises de IA.')
    GEMINI_API_KEY = None

In [ ]:
# CÉLULA 4 — Importações e configurações
import os, time, json, warnings, math
from datetime import datetime, timedelta
from io import StringIO
import pandas as pd
import requests
import yfinance as yf
from docx import Document
from docx.shared import Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from google import genai
warnings.filterwarnings('ignore')

# Configurações ajustáveis
MODELO_GEMINI          = 'gemini-2.5-flash'
FILTRO_LIQUIDEZ_MINIMA = 500_000
FILTRO_PL_MINIMO       = 0
FILTRO_PL_MAXIMO       = 40
FILTRO_DY_MINIMO       = 0.06
FILTRO_PVPA_MAXIMO     = 5.0
FILTRO_BESST_ATIVO     = True
TOP_RANKING            = 10
TOP_ANALISE            = 5
ANOS_HISTORICO         = 5
TICKERS_BESST = {
    'Bancos':     ['BBAS3','BBDC3','BBDC4','ITUB4','ITSA4','SANB11','BPAC11'],
    'Energia':    ['ELET3','ELET6','CMIG4','CPFE3','CPLE6','EGIE3','ENEV3','ENGI11','EQTL3','TAEE11','ISAE4'],
    'Saneamento': ['SBSP3'],
    'Seguros':    ['BBSE3','IRBR3','PSSA3'],
    'Telecom':    ['VIVT3','TIMS3'],
}
TODOS_TICKERS_BESST = [t for lista in TICKERS_BESST.values() for t in lista]
PESOS = {'dy':0.30,'roe':0.25,'pl':0.20,'pvpa':0.15,'div_ebitda':0.10}
COR_TITULO    = RGBColor(0x1F,0x49,0x7D)
COR_SUBTITULO = RGBColor(0x2E,0x75,0xB6)
COR_DESTAQUE  = RGBColor(0x70,0xAD,0x47)
COR_RISCO     = RGBColor(0xFF,0x00,0x00)
print('Configurações carregadas!')

In [ ]:
# CÉLULA 5 — Baixar e importar as funções do script principal
# O script completo fica no GitHub — esta célula baixa e importa
# todas as funções sem precisar duplicar o código aqui.
#
# SUBSTITUA a URL abaixo pelo link raw do seu repositório GitHub:
# Exemplo: https://raw.githubusercontent.com/SEU_USUARIO/SEU_REPO/main/relatorio_semanal_besst.py

GITHUB_RAW_URL = 'https://raw.githubusercontent.com/SEU_USUARIO/SEU_REPO/main/relatorio_semanal_besst.py'

import urllib.request
try:
    urllib.request.urlretrieve(GITHUB_RAW_URL, '/content/relatorio_semanal_besst.py')
    print('Script baixado do GitHub com sucesso!')
    # Importa as funções do script
    import importlib.util
    spec = importlib.util.spec_from_file_location('besst', '/content/relatorio_semanal_besst.py')
    mod  = importlib.util.load_from_spec(spec) if hasattr(importlib.util,'load_from_spec') else None
    print('Modulo carregado!')
except Exception as e:
    print(f'Nao foi possivel baixar do GitHub: {e}')
    print('Execute as células de funções manualmente abaixo.')

In [ ]:
# CÉLULA 6 — Executar pipeline completo e salvar relatório
# Esta célula coleta dados, gera o ranking, chama o Gemini
# e salva o .docx no Google Drive.

from google.colab import files as colab_files

inicio = datetime.now()
print('=== RELATORIO SEMANAL — CARTEIRA DE DIVIDENDOS ===')

# Coleta
print('\nColetando dados...')
tickers_ibov = buscar_ibovespa()
time.sleep(1)
df_todos = buscar_fundamentus()

# Filtros e ranking
df_filtrado = aplicar_filtros(df_todos, tickers_ibov)
if df_filtrado.empty:
    print('Nenhuma acao passou pelos filtros!')
else:
    df_ranking  = calcular_ranking(df_filtrado)
    tickers_hist = df_ranking.head(TOP_ANALISE).index.tolist()
    print(f'\nHistórico de dividendos para: {tickers_hist}')
    df_historico = buscar_historico_dividendos(tickers_hist)

    # Gemini
    print('\nInicializando Gemini...')
    cliente_gemini = inicializar_gemini()

    # Relatório
    print('\nGerando relatório Word...')
    doc = gerar_relatorio_word(df_ranking, df_historico, cliente_gemini)

    # Salva
    agora_str = datetime.now().strftime('%Y%m%d_%H%M')
    nome = f'relatorio_besst_{agora_str}.docx'
    caminho_drive = os.path.join(PASTA_DRIVE, nome)
    caminho_local = f'/content/{nome}'
    doc.save(caminho_drive)
    doc.save(caminho_local)

    tempo = (datetime.now() - inicio).seconds
    print(f'\nRelatório gerado em {tempo}s!')
    print(f'Salvo em: {caminho_drive}')
    print('Iniciando download...')
    colab_files.download(caminho_local)